# 02 - Modelado Predictivo, Validación Cruzada y Evaluación de Escenarios

Este notebook entrena, optimiza, valida y evalúa clasificadores supervisados para predecir el segmento de valor de departamentos en CABA (`bajo`, `medio`, `alto`).

### Modelos evaluados:
- **Línea Base (`dummy`):** Clasificador sin aprendizaje (`DummyClassifier`) según la estrategia definida en configuración para cuantificar la cota inferior de rendimiento.
- **Regresión Logística (`logreg`):** Modelo lineal con regularización L2 optimizado exhaustivamente mediante `GridSearchCV` sobre el espacio de hiperparámetros.
- **Random Forest (`random_forest`):** Ensamble de árboles de decisión optimizado mediante `RandomizedSearchCV` sobre una grilla multivariada.
- **XGBoost (`xgboost`):** Gradient Boosting implementado a través del wrapper `XGBTexto` para soportar directamente etiquetas de clase string, optimizado con `RandomizedSearchCV`.

### Metodología de validación:
1. **Partición Aleatoria:** Split 80/20 estratificado sobre todo el período temporal (2001-2016).
2. **Partición Temporal (Out-of-Time):** Entrenamiento en ventana histórica (2001-2013) y evaluación en años futuros (2014-2016).
3. **Validación Cruzada Estratificada:** 5 folds sobre el conjunto de entrenamiento para estimar estabilidad (media ± desvío).
4. **Auditoría de Data Leakage:** Comparativa controlada contra variantes que incorporan variables prohibidas (`DOLARES`, `AÑO`, `PRECIO_M2`).
5. **Persistencia en MongoDB Atlas:** Almacenamiento estructurado en colecciones `parametros`, `metricas`, `predicciones` y `validacion_cruzada`.

**Nota de ejecución:** Este notebook puede ejecutarse interactivamente o automatizado vía `python main.py` con inyección de parámetros mediante **Papermill**.

In [1]:
# Parámetros del notebook. Papermill (main.py) los reemplaza al ejecutarlo;
# si se corre a mano, valen estos valores.
search_sample_size = 20_000      # filas para buscar hiperparámetros
rf_n_iter = 8                    # combinaciones a probar en Random Forest
xgb_n_iter = 8                   # combinaciones a probar en XGBoost
cv_report_sample_size = 40_000   # filas para el informe de validación cruzada
models_to_run = ["logreg", "random_forest", "xgboost"]
run_temporal = True              # escenario temporal (train 2001-2013, test 2014-2016)
run_cv_report = True             # informe de validación cruzada (5 folds)
save_to_mongo = True             # guardar resultados en MongoDB
reuse_results = False            # True: reutiliza los modelos ya entrenados (models/<cache_name>)
cache_name = "results.joblib"
rf_trees_temporal = 100          # árboles del Random Forest en el escenario temporal
run_leakage = True               # prueba de fuga de información

## 1. Carga de datos limpios y partición estratificada

Se recupera la Tabla 1 desde la base relacional SQLite (`data/processed/deptos.db`). Se extraen las predictoras (`NUMERIC_FEATURES` y `CATEGORICAL_FEATURES`) asegurando la exclusión estricta de columnas prohibidas (`NON_FEATURES`), y se divide en 80% train y 20% test mediante partición estratificada por la variable objetivo.

In [2]:
import sys; sys.path.append("..")   # para encontrar config.py y src/ (este notebook está en notebooks/)
import config

# Los parámetros se aplican ANTES de importar src.models, que copia estos valores al importarse
config.SEARCH_SAMPLE_SIZE = search_sample_size
config.RF_N_ITER = rf_n_iter
config.XGB_N_ITER = xgb_n_iter
config.CV_REPORT_SAMPLE_SIZE = cv_report_sample_size

import joblib
import pandas as pd
from src.db_sql import load_clean_data
from src.features import get_xy, split_data
from src.models import tune_and_fit, fit_baseline, cv_report
from src.evaluation import evaluate_model
from src.temporal import temporal_split, fit_temporal
from src.db_mongo import save_params, save_metrics, save_predictions, save_leakage
from src.leakage_check import leakage_check
from src.logger_setup import flush_mongo_logs, get_logger

logger = get_logger("modelado")

# Datos limpios (Tabla 1, SQLite), separación predictoras/target y split 80/20 estratificado
df = load_clean_data()
X, y = get_xy(df)
X_train, X_test, y_train, y_test = split_data(X, y)

2026-11-19 05:54:46,558 | INFO     | src.db_sql | Tabla 'datos_limpios' leída: 174116 filas, 20 columnas
2026-11-19 05:54:50,135 | INFO     | src.features | X: (174116, 12) | y: (174116,) | predictoras: ['M2', 'AMBIENTES', 'ANTIGUEDAD', 'BAÑOS', 'BAULERA', 'COCHERA', 'LAVADERO', 'TERRAZA', 'LAT', 'LON', 'BARRIO', 'ORIENT']
2026-11-19 05:54:54,155 | INFO     | src.features | Train: 139292 filas | Test: 34824 filas
2026-11-19 05:54:54,889 | INFO     | src.features | Target en train:
SEGMENTO
alto     0.333
medio    0.333
bajo     0.333
2026-11-19 05:54:54,895 | INFO     | src.features | Target en test:
SEGMENTO
alto     0.333
medio    0.333
bajo     0.333


## 2. Prueba experimental de fuga de información (Data Leakage)

Se entrena y evalúa un clasificador Random Forest idéntico sobre 5 variantes de predictoras:
1. `sin DOLARES (modelo real)`: Predictoras legítimas (superficie, ambientes, barrio, etc.).
2. `sin DOLARES, con AÑO`: Evalúa si el año filtra la inflación temporal del target.
3. `con DOLARES`: Inyecta el precio absoluto en dólares.
4. `con DOLARES + AÑO`: Inyecta precio y año de publicación.
5. `con PRECIO_M2 + AÑO`: Inyecta la variable exacta a partir de la cual se construyeron los terciles del target.

Permite cuantificar la sobrestimación espuria de exactitud (delta de accuracy) provocada por la fuga de datos.

In [3]:
if run_leakage:
    leak = leakage_check(df)
    print(leak[["variante", "accuracy", "f1_macro", "delta_vs_modelo_real"]].round(3))
    if save_to_mongo:
        save_leakage(leak)
flush_mongo_logs()

2026-11-19 05:54:55,154 | INFO     | src.leakage_check | Leakage check: muestra de 40000 filas (train 32000, test 8000)
2026-11-19 05:55:18,735 | INFO     | src.leakage_check | Leakage check [sin DOLARES (modelo real)]: accuracy=0.647 f1_macro=0.644
2026-11-19 05:55:42,276 | INFO     | src.leakage_check | Leakage check [sin DOLARES, con AÑO]: accuracy=0.648 f1_macro=0.644
2026-11-19 05:55:56,152 | INFO     | src.leakage_check | Leakage check [con DOLARES]: accuracy=0.689 f1_macro=0.685
2026-11-19 05:56:00,147 | INFO     | src.leakage_check | Leakage check [con DOLARES + AÑO]: accuracy=0.709 f1_macro=0.705
2026-11-19 05:56:03,667 | INFO     | src.leakage_check | Leakage check [con PRECIO_M2 + AÑO]: accuracy=0.855 f1_macro=0.855


                    variante  accuracy  f1_macro  delta_vs_modelo_real
0  sin DOLARES (modelo real)     0.647     0.644                 0.000
1       sin DOLARES, con AÑO     0.648     0.644                 0.001
2                con DOLARES     0.689     0.685                 0.042
3          con DOLARES + AÑO     0.708     0.705                 0.061
4        con PRECIO_M2 + AÑO     0.855     0.855                 0.208


2026-11-19 05:56:05,737 | INFO     | src.db_mongo | Mongo 'leakage' [leakage]: 5 documentos insertados (run_id=20261119_055347)


## 3. Entrenamiento con búsqueda de hiperparámetros y caché en disco

Para optimizar los modelos de forma computacionalmente viable, la búsqueda de hiperparámetros (`GridSearchCV` para Regresión Logística y `RandomizedSearchCV` para árboles) se ejecuta sobre una submuestra estratificada de entrenamiento (`SEARCH_SAMPLE_SIZE`).

Una vez seleccionados los mejores hiperparámetros, el modelo final se reentrena sobre el **100% del conjunto de entrenamiento** (`refit=False` en la búsqueda). Los modelos ajustados se serializan en `models/results.joblib` para permitir su reutilización inmediata con la bandera `reuse_results=True`.

In [4]:
# Caché de resultados: entrenar el Random Forest tarda mucho. Con reuse_results=True se reutilizan
# los modelos ya guardados y solo se entrena lo que falte (por ejemplo, XGBoost si no estaba).
# Ojo: si cambian los datos, las predictoras o los hiperparámetros, correr con reuse_results=False.
config.MODELS_DIR.mkdir(exist_ok=True)
RES_PATH = config.MODELS_DIR / cache_name
cache = joblib.load(RES_PATH) if reuse_results and RES_PATH.exists() else {}

results, entrenados = {}, []
for nombre in models_to_run:
    if nombre in cache:
        logger.info("[%s] se reutiliza el modelo guardado en %s", nombre, RES_PATH.name)
        results[nombre] = cache[nombre]
    else:
        results[nombre] = tune_and_fit(nombre, X_train, y_train)
        entrenados.append(nombre)

# Línea base: ¿cuánto se logra sin aprender nada de las predictoras?
if "dummy" in cache:
    results["dummy"] = cache["dummy"]
else:
    results["dummy"] = fit_baseline(X_train, y_train)
    entrenados.append("dummy")

# Se actualiza la caché sin perder los modelos que ya estaban guardados
if entrenados:
    joblib.dump({**cache, **results}, RES_PATH)
    logger.info("Caché %s actualizada con: %s", RES_PATH.name, entrenados)

2026-11-19 05:56:06,497 | INFO     | src.models | [logreg] búsqueda de hiperparámetros sobre 20000 filas (GridSearchCV, cv=5, scoring=f1_macro)
2026-11-19 05:56:53,290 | INFO     | src.models | [logreg] mejores parámetros: {'clf__C': 1} | f1_macro en CV: 0.6116 | 47 s
2026-11-19 05:57:14,077 | INFO     | src.models | [logreg] reentrenado con 139292 filas en 21 s
2026-11-19 05:57:14,489 | INFO     | src.models | [random_forest] búsqueda de hiperparámetros sobre 20000 filas (RandomizedSearchCV, cv=5, scoring=f1_macro)
2026-11-19 06:13:22,996 | INFO     | src.models | [random_forest] mejores parámetros: {'clf__n_estimators': 200, 'clf__min_samples_leaf': 5, 'clf__max_features': 0.5, 'clf__max_depth': 20} | f1_macro en CV: 0.6444 | 969 s
2026-11-19 06:27:32,746 | INFO     | src.models | [random_forest] reentrenado con 139292 filas en 850 s
2026-11-19 06:27:33,449 | INFO     | src.models | [xgboost] búsqueda de hiperparámetros sobre 20000 filas (RandomizedSearchCV, cv=5, scoring=f1_macro)
2

## 4. Evaluación en escenario aleatorio (Split 80/20)

Cada modelo se evalúa sobre el 20% de test no visto. Se calculan exactitud (accuracy), precisión, recall y F1-score (promedios macro y ponderado), métricas desagregadas por clase y matriz de confusión completa.
Adicionalmente, se mide la exactitud sobre una submuestra de entrenamiento (`accuracy_train_sample`) para detectar oportunamente sobreajuste (gap > 5%).

In [5]:
all_metrics, all_preds = [], []
for name, r in results.items():
    m, p = evaluate_model(name, r["model"], X_train, y_train, X_test, y_test)
    all_metrics.append(m); all_preds.append(p)

resumen = pd.DataFrame(all_metrics)[["model", "accuracy", "precision_macro", "recall_macro",
                                     "f1_macro", "accuracy_train_sample"]].round(4)
print(resumen)

2026-11-19 06:30:40,287 | INFO     | src.evaluation | [logreg] TEST accuracy=0.6189 precision=0.6131 recall=0.6189 f1_macro=0.6151 (train acc 0.6222)
2026-11-19 06:30:40,289 | INFO     | src.evaluation | [logreg] matriz de confusión (filas=real, columnas=predicho ['bajo', 'medio', 'alto']):
[[8191 2560  856]
 [3316 5216 3075]
 [ 898 2565 8147]]
2026-11-19 06:30:45,759 | WARNING  | src.evaluation | [random_forest] posible sobreajuste: accuracy train 0.800 vs test 0.678
2026-11-19 06:30:46,074 | INFO     | src.evaluation | [random_forest] TEST accuracy=0.6784 precision=0.6777 recall=0.6784 f1_macro=0.6780 (train acc 0.8002)
2026-11-19 06:30:46,077 | INFO     | src.evaluation | [random_forest] matriz de confusión (filas=real, columnas=predicho ['bajo', 'medio', 'alto']):
[[8639 2452  516]
 [2668 6454 2485]
 [ 520 2559 8531]]
2026-11-19 06:30:50,961 | INFO     | src.evaluation | [xgboost] TEST accuracy=0.6686 precision=0.6661 recall=0.6686 f1_macro=0.6672 (train acc 0.6856)
2026-11-19 06:3

           model  accuracy  precision_macro  recall_macro  f1_macro  \
0         logreg    0.6189           0.6131        0.6189    0.6151   
1  random_forest    0.6784           0.6777        0.6784    0.6780   
2        xgboost    0.6686           0.6661        0.6686    0.6672   
3          dummy    0.3325           0.3325        0.3325    0.3325   

   accuracy_train_sample  
0                 0.6222  
1                 0.8002  
2                 0.6856  
3                 0.3324  


## 5. Validación cruzada estratificada (5 Folds)

Cada clasificador se evalúa a lo largo de 5 pliegues estratificados utilizando `cross_validate()` con los mejores hiperparámetros previamente encontrados.
Permite reportar la media ± desvío estándar de las métricas (`accuracy`, `precision_macro`, `recall_macro`, `f1_macro`), reflejando la estabilidad del modelo ante variaciones en la partición.

In [6]:
cv_df = pd.DataFrame()
if run_cv_report:
    cv_df = cv_report(results, X_train, y_train)
    metricas_cv = list(config.CV_REPORT_SCORING)

    # resultados por fold de cada modelo
    for nombre, g in cv_df.groupby("modelo", sort=False):
        print(f"\nModelo: {nombre}")
        display(g.set_index("fold")[metricas_cv].round(4))

    # resumen: media ± desvío entre folds
    media = cv_df.groupby("modelo", sort=False)[metricas_cv].mean()
    desvio = cv_df.groupby("modelo", sort=False)[metricas_cv].std(ddof=0)
    print("\nResumen (media ± desvío):")
    display(media.round(4).astype(str) + " ± " + desvio.round(4).astype(str))

2026-11-19 06:30:54,699 | INFO     | src.models | Informe de validación cruzada: 5 folds sobre 40000 filas, modelos: ['logreg', 'random_forest', 'xgboost', 'dummy']
2026-11-19 06:31:15,513 | INFO     | src.models | [logreg] CV 5 folds (21 s): accuracy=0.6182±0.0039, precision_macro=0.6124±0.0036, recall_macro=0.6182±0.0039, f1_macro=0.6144±0.0036
2026-11-19 06:42:24,344 | INFO     | src.models | [random_forest] CV 5 folds (669 s): accuracy=0.6538±0.0053, precision_macro=0.6529±0.0051, recall_macro=0.6538±0.0053, f1_macro=0.6532±0.0052
2026-11-19 06:42:50,241 | INFO     | src.models | [xgboost] CV 5 folds (26 s): accuracy=0.6553±0.0079, precision_macro=0.6524±0.0078, recall_macro=0.6553±0.0079, f1_macro=0.6536±0.0078
2026-11-19 06:42:52,090 | INFO     | src.models | [dummy] CV 5 folds (2 s): accuracy=0.3309±0.0019, precision_macro=0.3310±0.0019, recall_macro=0.3310±0.0019, f1_macro=0.3309±0.0019



Modelo: logreg


,accuracy,precision_macro,recall_macro,f1_macro
fold,,,,
1,0.6161,0.6113,0.6161,0.6128
2,0.6250,0.6189,0.6250,0.6208
3,0.6134,0.6082,0.6134,0.6101
4,0.6168,0.6106,0.6168,0.6128
5,0.6196,0.6132,0.6196,0.6154



Modelo: random_forest


,accuracy,precision_macro,recall_macro,f1_macro
fold,,,,
1,0.6491,0.6492,0.6491,0.6489
2,0.6632,0.6624,0.6632,0.6628
3,0.6494,0.6483,0.6494,0.6488
4,0.6514,0.6505,0.6514,0.6509
5,0.6558,0.6540,0.6558,0.6548



Modelo: xgboost


,accuracy,precision_macro,recall_macro,f1_macro
fold,,,,
1,0.6449,0.6420,0.6449,0.6432
2,0.6681,0.6652,0.6681,0.6665
3,0.6522,0.6501,0.6523,0.6510
4,0.6516,0.6489,0.6516,0.6501
5,0.6597,0.6559,0.6598,0.6574



Modelo: dummy


,accuracy,precision_macro,recall_macro,f1_macro
fold,,,,
1,0.3324,0.3325,0.3324,0.3324
2,0.3304,0.3303,0.3304,0.3303
3,0.3330,0.3331,0.3330,0.3330
4,0.3276,0.3277,0.3276,0.3276
5,0.3314,0.3314,0.3314,0.3313



Resumen (media ± desvío):


,accuracy,precision_macro,recall_macro,f1_macro
modelo,,,,
logreg,0.6182 ± 0.0039,0.6124 ± 0.0036,0.6182 ± 0.0039,0.6144 ± 0.0036
random_forest,0.6538 ± 0.0053,0.6529 ± 0.0051,0.6538 ± 0.0053,0.6532 ± 0.0052
xgboost,0.6553 ± 0.0079,0.6524 ± 0.0078,0.6553 ± 0.0079,0.6536 ± 0.0078
dummy,0.3309 ± 0.0019,0.331 ± 0.0019,0.331 ± 0.0019,0.3309 ± 0.0019


## 6. Evaluación en escenario temporal (Out-Of-Time)

Para evaluar la capacidad predictiva en un entorno de producción realista sin sesgo de contemporaneidad, se entrena con avisos publicados hasta el año de corte (`TEMPORAL_CUTOFF = 2013`) y se evalúa sobre los años posteriores (2014 a 2016).
Los hiperparámetros se heredan directamente del escenario aleatorio mediante clonación (`clone()`), midiendo la degradación anual del rendimiento.

In [7]:
t_metrics, t_preds = [], []
if run_temporal:
    Xtr_t, Xte_t, ytr_t, yte_t = temporal_split(df, X, y)
    for name, r in results.items():
        mdl = fit_temporal(name, r["model"], Xtr_t, ytr_t, rf_trees=rf_trees_temporal)
        m, p = evaluate_model(name, mdl, Xtr_t, ytr_t, Xte_t, yte_t)
        t_metrics.append(m); t_preds.append(p)

    print(pd.DataFrame(t_metrics)[["model", "accuracy", "precision_macro",
                                   "recall_macro", "f1_macro"]].round(4))
    # accuracy por año: muestra si algún año (por ejemplo 2016) se predice peor
    pt = pd.concat(t_preds)
    pt["AÑO"] = df.loc[pt["id_fila"], "AÑO"].to_numpy()
    print(pt.groupby(["modelo", "AÑO"])["acierto"].mean().unstack().round(3))

2026-11-19 06:42:55,937 | INFO     | src.temporal | Split temporal: train 145850 filas (2001-2013) | test 28266 filas (2014-2016)
2026-11-19 06:43:13,356 | INFO     | src.temporal | [logreg] ajuste temporal con 145850 filas en 17 s
2026-11-19 06:43:15,100 | INFO     | src.evaluation | [logreg] TEST accuracy=0.6078 precision=0.6016 recall=0.6078 f1_macro=0.6037 (train acc 0.6190)
2026-11-19 06:43:15,101 | INFO     | src.evaluation | [logreg] matriz de confusión (filas=real, columnas=predicho ['bajo', 'medio', 'alto']):
[[6645 2036  740]
 [2667 4123 2631]
 [ 855 2156 6413]]
2026-11-19 06:52:07,740 | INFO     | src.temporal | [random_forest] ajuste temporal con 145850 filas en 532 s
2026-11-19 06:52:10,465 | WARNING  | src.evaluation | [random_forest] posible sobreajuste: accuracy train 0.796 vs test 0.641
2026-11-19 06:52:10,671 | INFO     | src.evaluation | [random_forest] TEST accuracy=0.6411 precision=0.6414 recall=0.6411 f1_macro=0.6413 (train acc 0.7964)
2026-11-19 06:52:10,673 | IN

           model  accuracy  precision_macro  recall_macro  f1_macro
0         logreg    0.6078           0.6016        0.6078    0.6037
1  random_forest    0.6411           0.6414        0.6411    0.6413
2        xgboost    0.6373           0.6347        0.6373    0.6358
3          dummy    0.3334           0.3334        0.3334    0.3334
AÑO             2014   2015   2016
modelo                            
dummy          0.332  0.336  0.330
logreg         0.616  0.615  0.563
random_forest  0.656  0.647  0.581
xgboost        0.651  0.643  0.580


## 7. Persistencia estructurada en MongoDB Atlas

Se exportan todos los resultados experimentales a la base de datos no relacional MongoDB Atlas identificados por el `RUN_ID` actual:
- `parametros`: Configuración, hiperparámetros óptimos, tiempo de cómputo y score de CV (Tabla 3).
- `metricas`: Métricas globales, soporte por clase y matriz de confusión de ambos escenarios (Tabla 2).
- `predicciones`: Inferencia fila a fila con probabilidades por clase para cada modelo y escenario.
- `validacion_cruzada`: Métricas detalladas pliegue por pliegue.

In [8]:
if save_to_mongo:
    save_params(results)
    save_metrics(all_metrics, "aleatorio")
    save_predictions(pd.concat(all_preds, ignore_index=True), "aleatorio")
    if run_temporal:
        save_metrics(t_metrics, "temporal")
        save_predictions(pd.concat(t_preds, ignore_index=True), "temporal")
    if run_cv_report:
        from src.db_mongo import save_cv   # import local: solo hace falta si se guarda en Mongo
        save_cv(cv_df)
flush_mongo_logs()

2026-11-19 06:52:33,735 | INFO     | src.db_mongo | Mongo 'parametros' [comun]: 4 documentos insertados (run_id=20261119_055347)
2026-11-19 06:52:34,493 | INFO     | src.db_mongo | Mongo 'metricas' [aleatorio]: 4 documentos insertados (run_id=20261119_055347)
2026-11-19 06:54:53,001 | INFO     | src.db_mongo | Mongo 'predicciones' [aleatorio]: 139296 documentos insertados (run_id=20261119_055347)
2026-11-19 06:54:53,915 | INFO     | src.db_mongo | Mongo 'metricas' [temporal]: 4 documentos insertados (run_id=20261119_055347)
2026-11-19 06:56:40,564 | INFO     | src.db_mongo | Mongo 'predicciones' [temporal]: 113064 documentos insertados (run_id=20261119_055347)
2026-11-19 06:56:41,793 | INFO     | src.db_mongo | Mongo 'validacion_cruzada' [validacion_cruzada]: 20 documentos insertados (run_id=20261119_055347)


## Utilidad: vaciar colecciones de MongoDB (dejar comentado)

Solo para empezar de cero. Borra TODO lo de esas colecciones, de todas las corridas. No toca `logs`.

In [9]:
# from src.db_mongo import get_db
# db = get_db()
# for c in ("predicciones", "metricas", "parametros", "leakage", "validacion_cruzada"):
#     db[c].drop()

# Últimos Análisis

In [10]:
# comparar los dos modelos sobre las mismas filas de test, con la prueba de McNemar.
from scipy.stats import binomtest
from src.db_mongo import load_run, get_db

p = load_run("predicciones", "aleatorio")
a = p[p["modelo"] == "random_forest"].set_index("id_fila")["acierto"]
b = p[p["modelo"] == "xgboost"].set_index("id_fila")["acierto"].loc[a.index]

solo_rf = int((a & ~b).sum())      # filas que acierta solo el Random Forest
solo_xgb = int((~a & b).sum())     # filas que acierta solo XGBoost
print("Random Forest: ", solo_rf," - XGBoost: " ,solo_xgb, "p-valor:", binomtest(solo_rf, solo_rf + solo_xgb, 0.5).pvalue)

t = load_run("predicciones", "temporal")
a = t[t["modelo"] == "random_forest"].set_index("id_fila")["acierto"]
b = t[t["modelo"] == "xgboost"].set_index("id_fila")["acierto"].loc[a.index]

solo_rf = int((a & ~b).sum())      # filas que acierta solo el Random Forest
solo_xgb = int((~a & b).sum())     # filas que acierta solo XGBoost
print("Random Forest: ", solo_rf," - XGBoost: " ,solo_xgb, "p-valor:", binomtest(solo_rf, solo_rf + solo_xgb, 0.5).pvalue)

2026-11-19 06:56:51,392 | INFO     | src.db_mongo | Mongo 'predicciones' [aleatorio]: 139296 documentos leídos (run_id=20261119_055347)


Random Forest:  1945  - XGBoost:  1605 p-valor: 1.2420365336689621e-08


2026-11-19 06:56:57,856 | INFO     | src.db_mongo | Mongo 'predicciones' [temporal]: 113064 documentos leídos (run_id=20261119_055347)


Random Forest:  1444  - XGBoost:  1336 p-valor: 0.042401238589800154


In [11]:
db = get_db()
for c in ["parametros", "metricas", "predicciones", "leakage", "validacion_cruzada", "logs"]:
    print(c, db[c].count_documents({}))          # esperados: 4, 8, 252360, 5, 20, y logs

par = load_run("parametros", "comun")
print(par[["modelo", "mejores_parametros", "cv_score", "segundos_busqueda", "segundos_ajuste_final"]].to_string())

print(load_run("leakage", "leakage")[["variante", "accuracy", "f1_macro"]].round(3))

parametros 8
metricas 16
predicciones 504720
leakage 10
validacion_cruzada 40
logs 3169


2026-11-19 06:57:00,365 | INFO     | src.db_mongo | Mongo 'parametros' [comun]: 4 documentos leídos (run_id=20261119_055347)


          modelo                                                                                                               mejores_parametros  cv_score  segundos_busqueda  segundos_ajuste_final
0         logreg                                                                                                                    {'clf__C': 1}  0.611637               46.8                  20.80
1  random_forest                           {'clf__n_estimators': 200, 'clf__min_samples_leaf': 5, 'clf__max_features': 0.5, 'clf__max_depth': 20}  0.644390              968.5                 849.60
2        xgboost  {'clf__subsample': 0.8, 'clf__n_estimators': 100, 'clf__max_depth': 7, 'clf__learning_rate': 0.1, 'clf__colsample_bytree': 0.8}  0.649260              161.4                  15.50
3          dummy                                                                                                       {'strategy': 'stratified'}  0.333844                0.0                   0.33


2026-11-19 06:57:01,050 | INFO     | src.db_mongo | Mongo 'leakage' [leakage]: 5 documentos leídos (run_id=20261119_055347)


                    variante  accuracy  f1_macro
0  sin DOLARES (modelo real)     0.647     0.644
1       sin DOLARES, con AÑO     0.648     0.644
2                con DOLARES     0.689     0.685
3          con DOLARES + AÑO     0.708     0.705
4        con PRECIO_M2 + AÑO     0.855     0.855


In [12]:
# brecha entre train y test de este entrenamiento, y cómo se reparten los errores por clase en XGBoost.
m = load_run("metricas", "aleatorio")
print(m[["model", "accuracy", "accuracy_train_sample"]].round(4))
for _, r in m.iterrows():
    print(r["model"], {k: round(v["recall"], 3) for k, v in r["per_class"].items()}, r["confusion_matrix"])

2026-11-19 06:57:01,943 | INFO     | src.db_mongo | Mongo 'metricas' [aleatorio]: 4 documentos leídos (run_id=20261119_055347)


           model  accuracy  accuracy_train_sample
0         logreg    0.6189                 0.6222
1  random_forest    0.6784                 0.8002
2        xgboost    0.6686                 0.6856
3          dummy    0.3325                 0.3324
logreg {'bajo': 0.706, 'medio': 0.449, 'alto': 0.702} [[8191, 2560, 856], [3316, 5216, 3075], [898, 2565, 8147]]
random_forest {'bajo': 0.744, 'medio': 0.556, 'alto': 0.735} [[8639, 2452, 516], [2668, 6454, 2485], [520, 2559, 8531]]
xgboost {'bajo': 0.737, 'medio': 0.528, 'alto': 0.74} [[8558, 2462, 587], [2742, 6131, 2734], [528, 2487, 8595]]
dummy {'bajo': 0.338, 'medio': 0.332, 'alto': 0.328} [[3921, 3872, 3814], [3811, 3851, 3945], [3941, 3862, 3807]]
